In [1]:
import h2o
print(h2o.__version__)

3.46.0.9


In [ ]:
# H2O + LightGBM + Optuna + TF-IDF + SVD + GPU 적용
# Encoding -> TargetEncoder 적용
# LightGBM -> GPU with Optuna

In [11]:
import os
import re
import gc
import json
import datetime
import warnings

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ==============================
# JAVA (H2O 전용 설정)
# ==============================
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.29.7-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# ==============================
# H2O AutoML
# ==============================
import h2o
from h2o.automl import H2OAutoML

# ==============================
# NLP Vectorization
# ==============================
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD

# ==============================
# Model Eval Metrics
# ==============================
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    root_mean_squared_error = None

# ==============================
# GPU Models & Optimization
# ==============================
import optuna                          # Optuna Hyperparameter Tuning
import lightgbm as lgb                # GPU-enabled LightGBM
from category_encoders import TargetEncoder  # Target Encoding for Categorical Compression

# ==============================
# Explainability (Optional but used later)
# ==============================
import shap   # SHAP Explainability (TreeExplainer)

In [12]:
class MercariFullPipeline:

    def __init__(self, data_dir="../data", images_dir="../images", results_dir="../results"):
        self.data_dir = data_dir
        self.images_dir = images_dir
        self.results_dir = results_dir

        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(results_dir, exist_ok=True)

        self.train = None
        self.test = None
        self.train_vectorized = None
        self.test_vectorized = None

        self.train_hf = None
        self.test_hf = None
        self.automl = None
        self.best_model = None
        self.leaderboard_df = None

        self.enc = None
        self.lgbm_model = None
        self.lgbm_best_params = None


    # ---------------------- Utility ----------------------
    def _simple_normalize(self, text):
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        return re.sub(r"\s+", " ", text).strip()


    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        combined = pd.concat([self.train[col], self.test[col]])
        top_values = combined.value_counts().index[:top_k]
        self.train[col] = self.train[col].where(self.train[col].isin(top_values), rare_label)
        self.test[col] = self.test[col].where(self.test[col].isin(top_values), rare_label)


    # ---------------------- 1. Load ----------------------
    def load_data(self, train_file="train.tsv", test_file="test.tsv"):
        print("📂 Loading dataset...")

        self.train = pd.read_csv(os.path.join(self.data_dir, train_file), sep="\t")
        self.test = pd.read_csv(os.path.join(self.data_dir, test_file), sep="\t")

        self.train = self.train[self.train["price"] > 0]

        for df in [self.train, self.test]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: x.split("/") if isinstance(x, str) and "/" in x else ["missing"] * 3
                )
            )
            df["name"].fillna("unknown", inplace=True)
            df["brand_name"].fillna("Unknown", inplace=True)
            df["item_description"].fillna("No desc", inplace=True)

        self.train["price"] = np.log1p(self.train["price"].clip(1, 2500))

        self._collapse_rare_values("brand_name", 2000)
        self._collapse_rare_values("sub_cat", 300)
        self._collapse_rare_values("sub_sub_cat", 300)

        print("✔ Load complete.")


    # ---------------------- 2. Vectorize ----------------------
    def vectorize_text(self, method="tfidf", max_features=35000, n_components=80):
        print("🔤 TF-IDF + SVD Vectorizing...")

        vectors, names = [], []
        text_cols = ["name", "item_description"]

        for col in tqdm(text_cols):
            self.train[f"{col}_clean"] = self.train[col].apply(self._simple_normalize)
            self.test[f"{col}_clean"] = self.test[col].apply(self._simple_normalize)

            vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), dtype=np.float32)
            vec.fit(pd.concat([self.train[f"{col}_clean"], self.test[f"{col}_clean"]]))

            train_vec = vec.transform(self.train[f"{col}_clean"])
            test_vec = vec.transform(self.test[f"{col}_clean"])

            svd = TruncatedSVD(n_components=n_components, random_state=23)
            vectors.append((svd.fit_transform(train_vec), svd.transform(test_vec)))
            names.append([f"{col}_{i}" for i in range(n_components)])

        X_train = np.hstack([v[0] for v in vectors])
        X_test = np.hstack([v[1] for v in vectors])

        df_train = pd.DataFrame(X_train, columns=[col for sub in names for col in sub])
        df_test = pd.DataFrame(X_test, columns=[col for sub in names for col in sub])

        df_train["shipping"] = self.train["shipping"]
        df_test["shipping"] = self.test["shipping"]

        df_train["item_condition"] = self.train["item_condition_id"]
        df_test["item_condition"] = self.test["item_condition_id"]

        # Target encoding
        cat_cols = ["brand_name", "main_cat", "sub_cat", "sub_sub_cat"]
        self.enc = TargetEncoder(cols=cat_cols)

        self.train_vectorized = self.enc.fit_transform(df_train.join(self.train[cat_cols]), self.train["price"])
        self.test_vectorized = self.enc.transform(df_test.join(self.test[cat_cols]))

        print("✔ Vectorization complete:", self.train_vectorized.shape)


    # ---------------------- 3. H2O ----------------------
    def init_h2o(self, max_mem_size="12G"):
        print("🚀 Initializing H2O...")
        h2o.init(max_mem_size=max_mem_size)


    def train_automl(self, max_models=40, metric="RMSE", seed=23, nfolds=5):

        df = self.train_vectorized.copy()
        df["price"] = self.train["price"].values

        self.train_hf = h2o.H2OFrame(df)
        self.test_hf = h2o.H2OFrame(self.test_vectorized)

        for col in ["brand_name", "main_cat", "sub_cat", "sub_sub_cat", "shipping", "item_condition"]:
            if col in self.train_hf.columns:
                self.train_hf[col] = self.train_hf[col].asfactor()
                self.test_hf[col] = self.test_hf[col].asfactor()

        self.feature_cols = [col for col in self.train_hf.columns if col != "price"]

        self.automl = H2OAutoML(max_models=max_models, seed=seed, sort_metric=metric, nfolds=nfolds)
        self.automl.train(x=self.feature_cols, y="price", training_frame=self.train_hf)

        self.leaderboard_df = self.automl.leaderboard.as_data_frame()
        self.best_model = self.automl.leader

        print("🔥 H2O Leaderboard:")
        print(self.leaderboard_df.head())


    def use_stacked_ensemble_as_leader(self):
        se = self.leaderboard_df[self.leaderboard_df["model_id"].str.contains("Stacked")]
        if len(se):
            best = se.iloc[0]["model_id"]
            self.best_model = h2o.get_model(best)
            print(f"🔥 Using Stacked Ensemble: {best}")
        else:
            print("⚠ No Stacked Ensemble Model Found")


    # ---------------------- 4. LightGBM (GPU) ----------------------
    def train_lgbm_optuna(self, n_trials=20):
        print("🔥 Training LightGBM (GPU)...")

        X = self.train_vectorized.copy()
        y = self.train["price"].values
        y_true = np.expm1(y)

        X_train, X_valid, y_train, y_valid, _, y_valid_true = train_test_split(
            X, y, y_true, test_size=0.2, random_state=23
        )

        train_set = lgb.Dataset(X_train, label=y_train)
        valid_set = lgb.Dataset(X_valid, label=y_valid)

        def objective(trial):
            params = {
                "objective": "regression",
                "metric": "rmse",
                "learning_rate": trial.suggest_float("lr", 0.01, 0.15),
                "num_leaves": trial.suggest_int("num_leaves", 31, 255),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "device": "gpu",
                "gpu_platform_id": 0,
                "gpu_device_id": 0
            }
            model = lgb.train(params, train_set, valid_sets=[valid_set], callbacks=[lgb.early_stopping(15)])
            preds = np.expm1(model.predict(X_valid))
            return mean_squared_error(y_valid_true, preds, squared=False)

        import optuna
        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=n_trials)

        print("🔥 Best Params:", study.best_params)

        params = {"objective": "regression", "metric": "rmse", "device": "gpu"}
        params.update(study.best_params)

        self.lgbm_model = lgb.train(params, lgb.Dataset(X, label=y))
        self.lgbm_best_params = params


    # ---------------------- 5. SHAP ----------------------
    def analyze_shap(self, sample=3000):
        import shap

        X = self.train_vectorized.head(sample)
        explainer = shap.TreeExplainer(self.lgbm_model)
        shap_values = explainer.shap_values(X)

        print("🔥 SHAP Top Features:")
        importance = np.abs(shap_values).mean(0)
        idx = np.argsort(importance)[::-1][:20]

        for i in idx:
            print(f"{X.columns[i]} → {importance[i]:.4f}")


    # ---------------------- 6. Predict ----------------------
    def predict_test(self, file="submission.csv"):
        preds = self.best_model.predict(self.test_hf).as_data_frame()['predict'].values
        preds = np.expm1(preds)

        df = pd.DataFrame({"test_id": self.test["test_id"], "price": preds})
        df.to_csv(os.path.join(self.results_dir, file), index=False)

        print(f"📁 Submission Saved: {file}")

        return df


In [13]:
# ------------------ 실행 예시 ------------------


    
# MercariFullPipeline()

# load_data()
# vectorize_text()

# init_h2o()
# train_automl()
# use_stacked_ensemble_as_leader()

# train_lgbm_optuna(n_trials=20)
# analyze_shap()

# predict_test()


In [14]:
analyzer = MercariFullPipeline(
    data_dir="../data",
    images_dir="../images",
    results_dir="../results"
)

In [15]:
analyzer.load_data()

📂 Loading dataset...
✔ Load complete.


In [ ]:
analyzer.vectorize_text(
    method="tfidf",
    max_features=35000,
    n_components=80
)

In [ ]:
analyzer.init_h2o(max_mem_size="12G")
# analyzer.init_h2o(max_mem_size="16G") -> 에러났을 때 이걸로 변경

In [ ]:
analyzer.train_automl(
    max_models=40,
    metric="RMSE",
    seed=23,
    nfolds=5
)

In [ ]:
analyzer.use_stacked_ensemble_as_leader()

In [ ]:
analyzer.train_lgbm_optuna(n_trials=20)

In [ ]:
analyzer.analyze_shap(sample=3000)

In [ ]:
analyzer.predict_test("submission_h2o.csv")